# 🔩 Fine-Tuning for Steel Inspection with Few Images: Pre-trained ResNet18 vs. Training from Scratch (NEU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtzai/course_ai_for_industrial_control_students/blob/main/notebooks/03_steel_defects_classification/n17_steel_defect_neu_finetuning_resnet18.ipynb)

> **En Colab:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU**. With a T4 GPU the whole notebook takes roughly 5–10 min; on a laptop CPU it can take more than an hour (use `QUICK_RUN = True` to try the code first).

**P6 (II).** On a new production line you rarely have thousands of labelled defect images. Here we test how much a network
**pre-trained on ImageNet** helps when we only have **10, 50 or all (180)** images per defect class of the NEU Surface Defect Database.

We compare three strategies with the **same ResNet18 architecture**:

| Strategy | Initial weights | What is trained | Learning rate |
|---|---|---|---|
| 🧱 **From scratch** | random | the whole network | 1e-3 |
| 🧊 **Feature extractor** | ImageNet | only the new last layer (backbone frozen) | 1e-3 |
| 🔥 **Fine-tuning** | ImageNet | last layer + the two highest blocks (`layer3`, `layer4`) | 1e-3 head, **1e-4** backbone |

📚 Theory: ResNet (Parte 3, slides 50–68); data-efficient methods (Parte 3, slide 97); fine-tuning, *"feature extractor"*, smaller learning rate and freezing the first layers (Parte 3, slides 106–118; Parte 2, slides 132–133).

The split is **the same as in n12–n14** (blocks of consecutive images, seed 42): the small training sets are subsets of the same 180 training images per class, and validation/test never change.

In [ ]:
# ⚙️ Setup cell (run first). Detects Google Colab and installs only what may be missing there.
# Locally (uv / VS Code) this cell does nothing: all dependencies are already in pyproject.toml.
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub', 'gdown'], check=False)
print('Running in Colab' if IN_COLAB else 'Running locally')

In [ ]:
# 📦 Import Required Libraries
import os
import time
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import models
from torchvision.transforms import v2
from sklearn.metrics import confusion_matrix, accuracy_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
if device.type == 'cpu':
    print('⚠️ No GPU detected: consider QUICK_RUN = True or switching Colab to a GPU runtime.')

QUICK_RUN = os.environ.get('QUICK_RUN', '0') == '1'   # set to True to test the code quickly (smaller images, fewer epochs)
IMG_SIZE = 112 if QUICK_RUN else 224       # ResNet18 was pre-trained on 224x224 images
BATCH_SIZE = 32
NUM_WORKERS = 2 if IN_COLAB else 0   # parallel data loading in Colab; 0 is safest on Windows/Jupyter
EPOCHS = 2 if QUICK_RUN else 15            # minimum epochs per experiment ...
MIN_STEPS = 10 if QUICK_RUN else 150       # ... but at least MIN_STEPS weight updates (10 images/class = only 2 batches per epoch)
N_PER_CLASS_LIST = [10, 50, 'all']         # training images per class ('all' = 180)
REPEATS = 1                                # >1 repeats each experiment with a different random subset (mean ± std)
LR_HEAD, LR_BACKBONE, LR_SCRATCH = 1e-3, 1e-4, 1e-3
os.environ.setdefault('TORCH_HOME', str(os.path.abspath('./data/torch')))   # pre-trained weights are stored in ./data
print('QUICK_RUN =', QUICK_RUN, '| IMG_SIZE =', IMG_SIZE, '| EPOCHS =', EPOCHS, '| MIN_STEPS =', MIN_STEPS)

In [ ]:
# 📥 Download the NEU Surface Defect Database into ./data
# Sources tried in order (no Kaggle login needed for the first two):
#   1) kagglehub: 'kaustubhdikshit/neu-surface-defect-database' (public Kaggle mirror of NEU-DET)
#   2) Official Google Drive link of NEU-DET published by the dataset authors (via gdown)
#   3) Manual: download NEU-DET (or NEU-CLS) and unzip it anywhere inside ./data
import os
import zipfile
from pathlib import Path
from PIL import Image

DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True)
os.environ.setdefault('KAGGLEHUB_CACHE', str((DATA_DIR / 'kagglehub').resolve()))  # keep everything in ./data

NEU_CLASSES = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
NEU_KAGGLE_HANDLE = 'kaustubhdikshit/neu-surface-defect-database'
NEU_GDRIVE_ID = '1qrdZlaDi272eA79b0uCwwqPrm2Q_WI3k'   # NEU-DET.zip, official link from the authors' page


def list_neu_images(root):
    """All NEU images below `root`. Class and index come from the file name: <class>_<index>.jpg"""
    files = {}
    for p in Path(root).rglob('*.jpg'):
        cls, _, idx = p.stem.rpartition('_')
        if cls in NEU_CLASSES and idx.isdigit():
            files.setdefault(p.name, p)  # de-duplicate by file name
    return sorted(files.values())


def download_neu():
    if len(list_neu_images(DATA_DIR)) >= 1800:
        return DATA_DIR
    errors = []
    try:
        import kagglehub
        path = kagglehub.dataset_download(NEU_KAGGLE_HANDLE)
        if len(list_neu_images(path)) >= 1800:
            return Path(path)
    except Exception as e:  # noqa: BLE001
        errors.append(f'kagglehub: {e}')
    try:
        import gdown
        zip_path = DATA_DIR / 'NEU-DET.zip'
        if not zip_path.exists():
            gdown.download(id=NEU_GDRIVE_ID, output=str(zip_path), quiet=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(DATA_DIR)
        if len(list_neu_images(DATA_DIR)) >= 1800:
            return DATA_DIR
    except Exception as e:  # noqa: BLE001
        errors.append(f'Google Drive: {e}')
    raise RuntimeError(
        'Could not download NEU automatically.\n' + '\n'.join(errors) +
        '\nManual option: download NEU-DET from https://www.kaggle.com/datasets/' + NEU_KAGGLE_HANDLE +
        ' or from http://faculty.neu.edu.cn/songkc/en/zdylm/263270/list/index.htm and unzip it inside ./data')


NEU_ROOT = download_neu()
print(f'NEU images found: {len(list_neu_images(NEU_ROOT))} in {NEU_ROOT}')

In [ ]:
# 🧱 Load the images (grayscale, resized) and build a reproducible train / validation / test split
def load_neu(root, img_size):
    paths = list_neu_images(root)
    X = np.stack([np.array(Image.open(p).convert('L').resize((img_size, img_size), Image.BILINEAR))
                  for p in paths])                                       # (N, H, W) uint8
    y = np.array([NEU_CLASSES.index(p.stem.rpartition('_')[0]) for p in paths])
    idx = np.array([int(p.stem.rpartition('_')[2]) for p in paths])     # image number inside its class
    return X, y, idx


def blocked_split(y, idx, block_size=10, fractions=(0.6, 0.2, 0.2), seed=42):
    """Stratified split by blocks of consecutive image numbers.
    NEU images are crops of the same steel strips and neighbouring numbers can be very similar,
    so we keep each block of `block_size` consecutive images in ONE split (avoids optimistic test results)."""
    rng = np.random.default_rng(seed)
    parts = {'train': [], 'val': [], 'test': []}
    for c in np.unique(y):
        members = np.where(y == c)[0]
        blocks = (idx[members] - 1) // block_size
        order = rng.permutation(np.unique(blocks))
        n_tr = int(round(fractions[0] * len(order)))
        n_va = int(round(fractions[1] * len(order)))
        for name, sel in [('train', order[:n_tr]), ('val', order[n_tr:n_tr + n_va]), ('test', order[n_tr + n_va:])]:
            parts[name].extend(members[np.isin(blocks, sel)])
    return tuple(np.array(sorted(parts[k])) for k in ('train', 'val', 'test'))


X_all, y_all, idx_all = load_neu(NEU_ROOT, IMG_SIZE)
train_idx, val_idx, test_idx = blocked_split(y_all, idx_all, seed=SEED)
print('All images:', X_all.shape, '| classes:', NEU_CLASSES)
for name, ids in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    print(f'{name:>5}: {len(ids):4d} images | per class: {np.bincount(y_all[ids], minlength=6).tolist()}')

## 🔄 Input pipeline for an ImageNet network

ResNet18 expects **3-channel** images normalised with the ImageNet mean and std. Our images are gray, so we **repeat the gray channel 3 times**.
All strategies use the same light augmentation for training (horizontal/vertical flips) so that the only difference is the initialisation and what is trained.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
X_u8 = torch.from_numpy(X_all).unsqueeze(1)          # (N, 1, H, W) uint8
TRAIN_AUG = v2.Compose([v2.RandomHorizontalFlip(), v2.RandomVerticalFlip()])


class NEUImageNetDataset(Dataset):
    def __init__(self, ids, augment=False):
        self.ids, self.augment = np.asarray(ids), augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, k):
        i = self.ids[k]
        x = X_u8[i].float().div(255).repeat(3, 1, 1)   # gray -> 3 channels
        if self.augment:
            x = TRAIN_AUG(x)
        return (x - IMAGENET_MEAN) / IMAGENET_STD, int(y_all[i])


val_loader = DataLoader(NEUImageNetDataset(val_idx), batch_size=64, shuffle=False)
test_loader = DataLoader(NEUImageNetDataset(test_idx), batch_size=64, shuffle=False)


def subset_per_class(ids, n_per_class, seed):
    if n_per_class == 'all':
        return np.asarray(ids)
    rng = np.random.default_rng(seed)
    return np.sort(np.concatenate([rng.choice(ids[y_all[ids] == c], n_per_class, replace=False)
                                   for c in range(len(NEU_CLASSES))]))

## 🧠 The three models

`torchvision.models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)` downloads the ImageNet weights (~45 MB) the first time.
We replace the last layer `fc` (1000 ImageNet classes) by a new `Linear(512, 6)`.

In [ ]:
NUM_CLASSES = len(NEU_CLASSES)
WEIGHTS = models.ResNet18_Weights.IMAGENET1K_V1


def build_resnet(strategy):
    model = models.resnet18(weights=None if strategy == 'scratch' else WEIGHTS)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)      # new head, randomly initialised
    if strategy == 'scratch':
        params = [{'params': model.parameters(), 'lr': LR_SCRATCH}]
    elif strategy == 'feature_extractor':
        for p in model.parameters():
            p.requires_grad = False
        for p in model.fc.parameters():
            p.requires_grad = True
        params = [{'params': model.fc.parameters(), 'lr': LR_HEAD}]
    elif strategy == 'fine_tuning':
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith(('layer3', 'layer4', 'fc'))   # first layers stay frozen (generic edges/textures)
        params = [{'params': [p for n, p in model.named_parameters() if n.startswith(('layer3', 'layer4'))], 'lr': LR_BACKBONE},
                  {'params': model.fc.parameters(), 'lr': LR_HEAD}]
    else:
        raise ValueError(strategy)
    return model.to(device), params


for s in ['scratch', 'feature_extractor', 'fine_tuning']:
    m, _ = build_resnet(s)
    print(f'{s:<18} trainable parameters: {sum(p.numel() for p in m.parameters() if p.requires_grad):>10,} '
          f'of {sum(p.numel() for p in m.parameters()):,}')
del m

## 🧊 Feature extractor: compute the features once

When the backbone is **frozen**, its output for an image never changes. So we can pass every image through the backbone **once**,
store the 512-dimensional feature vectors and train only the last layer on them, which is extremely fast.
(For this reason the feature-extractor strategy does not use the flips.)

In [ ]:
@torch.no_grad()
def extract_features(ids):
    backbone = models.resnet18(weights=WEIGHTS)
    backbone.fc = nn.Identity()
    backbone.eval().to(device)
    loader = DataLoader(NEUImageNetDataset(ids), batch_size=64, shuffle=False)
    return torch.cat([backbone(xb.to(device)).cpu() for xb, _ in loader])


start = time.time()
all_ids = np.arange(len(y_all))
FEATURES = extract_features(all_ids)          # (1800, 512)
print(f'Features: {tuple(FEATURES.shape)} computed in {time.time() - start:.1f} s')

## 🏋️ Training functions
The CNN strategies use Adam with a **cosine learning-rate schedule** (the learning rate decays to ~0 at the end) and we evaluate the final weights.
The validation set is reported next to the test set so you can check that both tell the same story.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(device)).argmax(1).cpu()); labels.append(yb)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()


def set_train_mode(model, strategy):
    model.train()
    if strategy == 'fine_tuning':          # frozen blocks keep their ImageNet BatchNorm statistics
        for name in ['conv1', 'bn1', 'layer1', 'layer2']:
            getattr(model, name).eval()


def train_cnn(strategy, train_ids):
    model, params = build_resnet(strategy)
    optimizer = torch.optim.Adam(params, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(NEUImageNetDataset(train_ids, augment=True), batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS)
    epochs = max(EPOCHS, math.ceil(MIN_STEPS / len(loader)))
    # cosine schedule: the learning rate decays smoothly to ~0, so the final weights are stable (no early stopping needed)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(loader))
    for _epoch in range(epochs):
        set_train_mode(model, strategy)
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
    return model, epochs


def train_feature_head(train_ids):
    head = nn.Linear(FEATURES.shape[1], NUM_CLASSES).to(device)
    optimizer = torch.optim.Adam(head.parameters(), lr=LR_HEAD, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(FEATURES[train_ids], torch.from_numpy(y_all[train_ids]).long()),
                        batch_size=BATCH_SIZE, shuffle=True)
    epochs = max(50, math.ceil(500 / len(loader)))   # cheap: only 512 x 6 weights
    for _epoch in range(epochs):
        head.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(head(xb.to(device)), yb.to(device))
            loss.backward()
            optimizer.step()
    return head, epochs


@torch.no_grad()
def predict_head(head, ids):
    head.eval()
    return head(FEATURES[ids].to(device)).argmax(1).cpu().numpy(), y_all[ids]

## 🧪 Run all the experiments
3 strategies × 3 training-set sizes (× `REPEATS`). Each line prints the validation and test accuracy and the training time.

In [ ]:
STRATEGIES = ['scratch', 'feature_extractor', 'fine_tuning']
results, predictions = [], {}
total_start = time.time()
for rep in range(REPEATS):
    for n in N_PER_CLASS_LIST:
        train_ids = subset_per_class(train_idx, n, seed=SEED + rep)
        for strategy in STRATEGIES:
            torch.manual_seed(SEED + rep)
            start = time.time()
            if strategy == 'feature_extractor':
                head, epochs = train_feature_head(train_ids)
                val_pred, val_true = predict_head(head, val_idx)
                test_pred, test_true = predict_head(head, test_idx)
            else:
                model, epochs = train_cnn(strategy, train_ids)
                val_pred, val_true = predict(model, val_loader)
                test_pred, test_true = predict(model, test_loader)
                del model
            elapsed = time.time() - start
            res = {'strategy': strategy, 'images/class': n, 'train images': len(train_ids), 'repeat': rep,
                   'epochs': epochs, 'val acc': accuracy_score(val_true, val_pred),
                   'test acc': accuracy_score(test_true, test_pred), 'time (s)': elapsed}
            results.append(res)
            predictions[(strategy, n, rep)] = (test_true, test_pred)
            print(f'{strategy:<18} {str(n):>4}/class  epochs={epochs:3d}  val acc={res["val acc"]:.3f}  '
                  f'test acc={res["test acc"]:.3f}  time={elapsed:6.1f}s')
print(f'\nTotal experiment time: {(time.time() - total_start) / 60:.1f} min')

## 📊 Results: table and accuracy vs. number of images

In [ ]:
res_df = pd.DataFrame(results)
res_df['n'] = res_df['images/class'].map(lambda v: 180 if v == 'all' else v)
summary = res_df.groupby(['strategy', 'n']).agg(test_acc=('test acc', 'mean'), test_std=('test acc', 'std'),
                                                 val_acc=('val acc', 'mean'), time_s=('time (s)', 'mean')).reset_index()
print(summary.round(3).to_string(index=False))
print()
print(summary.pivot(index='strategy', columns='n', values='test_acc').round(3))

plt.figure(figsize=(7, 4.5))
for strategy, marker in zip(STRATEGIES, ['s', 'o', '^']):
    d = summary[summary.strategy == strategy]
    plt.errorbar(d['n'], d['test_acc'], yerr=d['test_std'].fillna(0), marker=marker, capsize=3, label=strategy)
plt.xscale('log'); plt.xticks([10, 50, 180], ['10', '50', '180 (all)'])
plt.axhline(1 / NUM_CLASSES, color='gray', ls=':', label='random guess')
plt.xlabel('Training images per class (log scale)'); plt.ylabel('Test accuracy')
plt.title('NEU: accuracy vs. number of training images'); plt.ylim(0, 1.02)
plt.grid(alpha=0.3, which='both'); plt.legend()
plt.tight_layout()
plt.show()

## 🔍 Confusion matrices with the smallest training set
With very few images, *where* each strategy fails is as informative as *how much*.

In [ ]:
n_small = N_PER_CLASS_LIST[0]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, strategy in zip(axes, STRATEGIES):
    t, p = predictions[(strategy, n_small, 0)]
    sns.heatmap(confusion_matrix(t, p), annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=NEU_CLASSES, yticklabels=NEU_CLASSES)
    ax.set_title(f'{strategy} - {n_small} images/class\ntest acc = {accuracy_score(t, p):.3f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 📝 Summary and Next Steps

- A network pre-trained on **ImageNet** (photos of objects) already contains generic filters (edges, textures, blobs) that are useful for **steel surfaces**, even though the domain is very different.
- **Feature extractor**: very few trainable parameters → fast and hard to overfit, but the features are not adapted to steel.
- **Fine-tuning**: adapts the high-level blocks with a **small learning rate** so the pre-trained knowledge is not destroyed; the first blocks stay frozen.
- **From scratch**: millions of parameters learned from a few hundred images → usually the worst when data is scarce.
- Look at **your** table: how does the gap between strategies change from 10 to 180 images per class? How long did each strategy take?

🔧 Things you can try: `REPEATS = 3` (how stable are the results with 10 images?), unfreeze also `layer2`, use the same learning rate (1e-3) for the backbone, or remove the flips.